# Cleaning V3 Built-in Operators Validation

这个 Notebook 验证 V3 清洗平台的最小闭环：`ParameterComputer + ImageBatch + LogicalOperatorSpec`，以及第一批内置算子：格式、尺寸、基础质量、空白图和完全重复检测。


## 1. 准备本地测试数据

In [ ]:
import json
import shutil
from io import BytesIO
from pathlib import Path

import pandas as pd
from PIL import Image

from image_gallery.cleaning import BasicCleaner
from image_gallery.dataset import Dataset
from image_gallery.operators.builtin import create_default_registry


repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

LIBRARY_ROOT = repo_root / "notebooks" / ".operators_test_library" / "cleaning_v3_builtin"
FIXTURE_DIR = LIBRARY_ROOT / "fixtures"
OUTPUT_DIR = LIBRARY_ROOT / "outputs"

if LIBRARY_ROOT.exists():
    shutil.rmtree(LIBRARY_ROOT)
FIXTURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_png_bytes(size: tuple[int, int], color: tuple[int, int, int], patterned: bool = True) -> bytes:
    image = Image.new("RGB", size, color=color)
    if patterned:
        for x in range(size[0]):
            for y in range(size[1]):
                if (x + y) % 2 == 0:
                    image.putpixel((x, y), (180, 60, 90))
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return buffer.getvalue()


ok_path = FIXTURE_DIR / "ok.png"
duplicate_path = FIXTURE_DIR / "duplicate.png"
small_path = FIXTURE_DIR / "small.png"
blank_path = FIXTURE_DIR / "blank.png"
bad_path = FIXTURE_DIR / "bad.png"
ok_path.write_bytes(make_png_bytes((32, 32), (80, 120, 160)))
duplicate_path.write_bytes(ok_path.read_bytes())
small_path.write_bytes(make_png_bytes((4, 4), (220, 120, 80)))
blank_path.write_bytes(make_png_bytes((32, 32), (255, 255, 255), patterned=False))
bad_path.write_bytes(b"not an image")

raw_path = LIBRARY_ROOT / "raw.parquet"
pd.DataFrame(
    {
        "image_id": ["ok", "dupe", "small", "blank", "bad"],
        "image_uri": [str(ok_path), str(duplicate_path), str(small_path), str(blank_path), str(bad_path)],
    }
).to_parquet(raw_path, index=False)

raw_path, [ok_path, duplicate_path, small_path, blank_path, bad_path]


## 2. 验证默认注册表

In [ ]:
registry = create_default_registry()

assert registry.list_operators() == [
    "content.blank_image_check",
    "duplicate.exact_duplicate_check",
    "format.decode_check",
    "quality.blur_check",
    "quality.brightness_check",
    "quality.contrast_check",
    "size.aspect_ratio_check",
    "size.dimension_check",
    "size.megapixel_check",
]

decode_spec = registry.get_operator("format.decode_check")
dimension_spec = registry.get_operator("size.dimension_check")
exact_duplicate_spec = registry.get_operator("duplicate.exact_duplicate_check")
assert decode_spec.required_parameters == ["decode_ok", "decode_error"]
assert dimension_spec.required_parameters == ["width", "height"]
assert exact_duplicate_spec.required_parameters == ["exact_duplicate_group_id", "exact_duplicate_count"]

computers = registry.find_computers_for_parameters(
    {
        "decode_ok",
        "decode_error",
        "width",
        "height",
        "aspect_ratio",
        "megapixels",
        "blur_score",
        "brightness_score",
        "contrast_score",
        "blank_score",
        "exact_duplicate_group_id",
        "exact_duplicate_count",
    }
)
assert [computer.name for computer in computers] == [
    "image_metadata_computer",
    "table_derived_computer",
    "image_quality_computer",
    "image_hash_computer",
    "duplicate_group_computer",
]

registry.list_operators()


## 3. 运行 BasicCleaner 并验证共享读取

In [ ]:
class CountingReadDataset(Dataset):
    def __init__(self, dataset_path: str) -> None:
        super().__init__(dataset_path=dataset_path)
        self.read_calls: list[str] = []

    def read_image_bytes(self, image_uri: str) -> bytes:
        self.read_calls.append(image_uri)
        return super().read_image_bytes(image_uri)


OPERATOR_CONFIGS = [
    {"format.decode_check": {"action": "drop"}},
    {"size.dimension_check": {"min_width": 8, "min_height": 8, "action": "drop"}},
    {"size.aspect_ratio_check": {}},
    {"size.megapixel_check": {"min_megapixels": 0.00001}},
    {"quality.blur_check": {"min_score": 0.0}},
    {"quality.brightness_check": {}},
    {"quality.contrast_check": {"min_score": 0.0}},
    {"content.blank_image_check": {}},
    {"duplicate.exact_duplicate_check": {}},
]

dataset = CountingReadDataset(str(raw_path))
cleaner = BasicCleaner(OPERATOR_CONFIGS)
cleaner.run(dataset, output_dir=OUTPUT_DIR)

context = cleaner._context
if context is None:
    raise AssertionError("cleaner context should exist")
paths = context.paths

assert dataset.read_calls == [str(ok_path), str(duplicate_path), str(small_path), str(blank_path), str(bad_path)]
assert paths.parameter_table_path.exists()
assert paths.evaluation_table_path.exists()
assert paths.operator_outputs_path.exists()
assert paths.parameter_manifest_path.exists()
assert (paths.relations_dir / "duplicate_pairs.parquet").exists()
assert paths.state_path.exists()

parameter_table = pd.read_parquet(paths.parameter_table_path)
evaluation_table = pd.read_parquet(paths.evaluation_table_path)
parameter_manifest = json.loads(paths.parameter_manifest_path.read_text(encoding="utf-8"))

parameter_table, evaluation_table, parameter_manifest


## 4. 验证参数表、评估表和 manifest

In [ ]:
required_parameter_columns = {
    "image_id",
    "image_uri",
    "width",
    "height",
    "decode_ok",
    "decode_error",
    "aspect_ratio",
    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "blank_score",
    "content_hash",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
}
assert required_parameter_columns.issubset(parameter_table.columns)

required_evaluation_columns = {
    "image_id",
    "image_uri",
    "decode_action",
    "decode_reason",
    "dimension_action",
    "dimension_reason",
    "aspect_ratio_action",
    "megapixel_action",
    "blur_action",
    "brightness_action",
    "contrast_action",
    "blank_action",
    "exact_duplicate_action",
    "exact_duplicate_reason",
    "final_action",
    "final_reason",
    "triggered_operator_names",
}
assert required_evaluation_columns.issubset(evaluation_table.columns)

for parameter_name in ["decode_ok", "decode_error", "width", "height"]:
    assert parameter_manifest[parameter_name]["computer"] == "image_metadata_computer"
    assert parameter_manifest[parameter_name]["stage"] == "image_batch"

expected_columns = {
    "aspect_ratio",
    "megapixels",
    "blur_score",
    "brightness_score",
    "contrast_score",
    "blank_score",
    "content_hash",
    "exact_duplicate_group_id",
    "exact_duplicate_count",
}
missing_columns = expected_columns - set(parameter_table.columns)
assert not missing_columns, sorted(missing_columns)

rows = evaluation_table.set_index("image_id")
assert rows.loc["ok", "final_action"] == "keep"
assert rows.loc["small", "dimension_action"] == "drop"
assert rows.loc["blank", "blank_action"] == "drop"
assert rows.loc["bad", "decode_action"] == "drop"
assert rows.loc["dupe", "exact_duplicate_action"] == "drop"

assert cleaner.preview().total_count == 5
assert cleaner.preview().clean_count == 1
assert cleaner.preview().review_count == 0
assert cleaner.preview().dropped_count == 4

cleaner.result("format.decode_check"), cleaner.result("duplicate.exact_duplicate_check")


## 5. 验证 rerun 只重算 evaluation

In [ ]:
before_parameters = pd.read_parquet(paths.parameter_table_path)

cleaner.config([
    {"size.dimension_check": {"min_width": 1, "min_height": 1, "action": "drop"}}
])
stale_state = cleaner.state()
assert stale_state.loc[stale_state["operator_name"] == "size.dimension_check", "status"].item() == "stale"

cleaner.rerun([
    {"size.dimension_check": {"min_width": 1, "min_height": 1, "action": "drop"}}
])
after_parameters = pd.read_parquet(paths.parameter_table_path)
assert before_parameters.equals(after_parameters)

after_rows = cleaner.export("full", str(LIBRARY_ROOT / "full_after_rerun.parquet")).to_frame().set_index("image_id")
assert after_rows.loc["ok", "final_action"] == "keep"
assert after_rows.loc["small", "final_action"] == "keep"
assert after_rows.loc["blank", "final_action"] == "drop"
assert after_rows.loc["bad", "final_action"] == "drop"
assert after_rows.loc["dupe", "final_action"] == "drop"
assert cleaner.preview().clean_count == 2
assert cleaner.preview().review_count == 0
assert cleaner.preview().dropped_count == 3

cleaner.state()


## 6. 验证导出视图

In [ ]:
full = cleaner.export("full", str(LIBRARY_ROOT / "full.parquet"))
clean = cleaner.export("clean", str(LIBRARY_ROOT / "clean.parquet"))
review = cleaner.export("review", str(LIBRARY_ROOT / "review.parquet"))
dropped = cleaner.export("dropped", str(LIBRARY_ROOT / "dropped.parquet"))
parameters = cleaner.export("parameters", str(LIBRARY_ROOT / "parameters.parquet"))
evaluations = cleaner.export("evaluations", str(LIBRARY_ROOT / "evaluations.parquet"))

assert full.count() == 5
assert clean.count() == 2
assert review.count() == 0
assert dropped.count() == 3
assert parameters.count() == 5
assert evaluations.count() == 5

print("PASS: cleaning v3 first operator batch notebook validation completed")
